<a href="https://colab.research.google.com/github/RatchanonPa/Data-Warehouse-and-Big-Data-Analytics/blob/main/Hackathon_5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# --- Mount Google Drive (Optional, if your data is on Drive) ---
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# prompt: un zip /content/drive/MyDrive/io-t-sleep-stage-classification-version-2.zip

!unzip /content/drive/MyDrive/io-t-sleep-stage-classification-version-2.zip

In [1]:
import pandas as pd
import glob
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.preprocessing import LabelEncoder, StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, LSTM, Dense, Dropout, BatchNormalization, MaxPooling1D, Input
from tensorflow.keras.callbacks import EarlyStopping, Callback
from sklearn.metrics import f1_score
from sklearn.utils import class_weight  # For class weights
# from imblearn.over_sampling import SMOTE  # For SMOTE (optional)


# --- Custom F1-Score Metric (Optional but Recommended) ---
class F1Score(tf.keras.metrics.Metric):
    def __init__(self, name='f1_score', **kwargs):
        super().__init__(name=name, **kwargs)
        self.precision = tf.keras.metrics.Precision()
        self.recall = tf.keras.metrics.Recall()

    def update_state(self, y_true, y_pred, sample_weight=None):
        y_pred = tf.argmax(y_pred, axis=1)
        y_true = tf.argmax(y_true, axis=1)
        self.precision.update_state(y_true, y_pred, sample_weight)
        self.recall.update_state(y_true, y_pred, sample_weight)

    def result(self):
        precision = self.precision.result()
        recall = self.recall.result()
        return 2 * ((precision * recall) / (precision + recall + tf.keras.backend.epsilon()))

    def reset_state(self):
        self.precision.reset_state()
        self.recall.reset_state()

# --- Custom Callback ---
class ValidationF1Callback(Callback):
    def __init__(self, validation_data):
        super().__init__()
        self.validation_data = validation_data

    def on_epoch_end(self, epoch, logs=None):
        X_val, y_val = self.validation_data
        y_pred = self.model.predict(X_val, verbose=0)
        y_pred_classes = np.argmax(y_pred, axis=1)
        y_true = np.argmax(y_val, axis=1)
        val_f1 = f1_score(y_true, y_pred_classes, average='weighted') # Use 'weighted' for imbalanced classes
        print(f" - val_f1_score: {val_f1:.4f}")
        logs['val_f1_score'] = val_f1


# --- Download Data (Kaggle API) ---
# !pip install kaggle
# !mkdir ~/.kaggle
# !cp /content/kaggle.json ~/.kaggle/  # Or /content/drive/MyDrive/...
# !chmod 600 ~/.kaggle/kaggle.json
# !kaggle competitions download -c io-t-sleep-stage-classification-version-2
# !unzip io-t-sleep-stage-classification-version-2.zip -d io-t-sleep-stage-classification-version-2

# --- Data Loading and Preprocessing (Train) ---

train_path = "/content/train/train"
train_csv_files = sorted(glob.glob(f"{train_path}/*.csv"))  # Sort for consistent order

if not train_csv_files:
    raise FileNotFoundError(f"No CSV files found in '{train_path}'.")

train_data = []
train_labels = []

for file in train_csv_files:
    df = pd.read_csv(file)
    target_col = "Sleep_Stage"
    feature_cols = [col for col in df.columns if col != target_col]
    train_data.append(df[feature_cols].values)
    train_labels.extend(df[target_col].values[::480])  # Get labels *before* reshaping

# Concatenate training data *before* reshaping.
X = np.concatenate(train_data, axis=0)
y = np.array(train_labels)  # Convert labels to NumPy array

# --- Data Scaling (BEFORE reshaping) ---
scaler = StandardScaler()
X = scaler.fit_transform(X) # Fit and transform on the *entire* training set

# Reshape X to 480 timesteps
num_samples = len(X) // 480
X = X[: num_samples * 480].reshape(num_samples, 480, -1)
# y = y[::480] # Labels are already correctly extracted

# Label Encoding and One-Hot Encoding
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)
num_classes = len(np.unique(y_encoded))
y_categorical = keras.utils.to_categorical(y_encoded, num_classes)

X = X.astype('float32')
y_categorical = y_categorical.astype('float32')

print("X shape:", X.shape)  # (num_samples, 480, num_features)
print("y_categorical shape:", y_categorical.shape)  # (num_samples, num_classes)
num_features = X.shape[2]

# --- Model Definition (CNN-LSTM) ---
# Use Input layer for explicit input shape definition
model = Sequential([
    Input(shape=(480, num_features)),  # Use Input layer
    Conv1D(filters=32, kernel_size=3, activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling1D(pool_size=2),
    Conv1D(filters=64, kernel_size=3, activation='relu', padding='same'),
    BatchNormalization(),
    MaxPooling1D(pool_size=2),
    Dropout(0.3),
    LSTM(64, return_sequences=True),
    LSTM(64),
    Dropout(0.3),
    Dense(128, activation='relu'),
    Dropout(0.3),
    Dense(num_classes, activation='softmax')
])


model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy', F1Score()])
model.summary()

# --- Training ---

split_ratio = 0.8
split_index = int(len(X) * split_ratio)
X_train = X[:split_index]
y_train = y_categorical[:split_index]
X_val = X[split_index:]
y_val = y_categorical[split_index:]

# --- Class Weights (Optional) ---
class_weights = class_weight.compute_class_weight('balanced', classes=np.unique(y_encoded), y=y_encoded)
class_weights_dict = dict(enumerate(class_weights))

# --- SMOTE (Optional - Uncomment if you want to use it) ---
# from imblearn.over_sampling import SMOTE
# smote = SMOTE(random_state=42)
# X_train_resampled, y_train_resampled = smote.fit_resample(X_train.reshape(X_train.shape[0], -1), y_encoded[:split_index])
# X_train = X_train_resampled.reshape(X_train_resampled.shape[0], 480, num_features)
# y_train = keras.utils.to_categorical(y_train_resampled, num_classes)


early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
val_f1_callback = ValidationF1Callback((X_val, y_val))

history = model.fit(
    X_train, y_train,
    epochs=100,
    batch_size=64,
    validation_data=(X_val, y_val),
    shuffle=False,  # Crucial for time series
    callbacks=[early_stopping, val_f1_callback],
    # class_weight=class_weights_dict  # Uncomment if using class weights
)

# --- Prediction (Corrected ID Generation and Concatenation) ---

test_base_path = "/content/test_segment/test_segment"
test_dirs = sorted(glob.glob(f"{test_base_path}/test*"))

if not test_dirs:
    raise FileNotFoundError(f"No test directories found in '{test_base_path}'.")

all_data = []
all_ids = []

for test_dir in test_dirs:
    test_csv_files = sorted(glob.glob(f"{test_dir}/*.csv"))
    if not test_csv_files:
        print(f"Warning: No CSV files found in '{test_dir}'. Skipping.")
        continue

    dir_data = []
    for file in test_csv_files:
        df_test = pd.read_csv(file)
        dir_data.append(df_test[feature_cols].values)

    dir_data = np.concatenate(dir_data, axis=0)
    print(f"Shape of dir_data BEFORE padding ({test_dir}):", dir_data.shape)

    num_rows = len(dir_data)
    remainder = num_rows % 480
    if remainder != 0:
        padding_rows = 480 - remainder
        padding_array = np.zeros((padding_rows, dir_data.shape[1]))
        dir_data = np.concatenate([dir_data, padding_array], axis=0)
    print(f"Shape of dir_data AFTER padding ({test_dir}):", dir_data.shape)

    file_prefix = test_dir.split('/')[-1]
    num_segments = len(dir_data) // 480
    dir_ids = [f"{file_prefix}_{i:05d}" for i in range(num_segments)]  # Correct, zero-padded IDs
    # dir_ids = [id_val for id_val in dir_ids for _ in range(480)] # expand id  <- REMOVE THIS LINE

    print(f"Length of dir_ids ({test_dir}):", len(dir_ids))
    assert len(dir_ids) == num_segments, f"ID and segment count mismatch in {test_dir}"

    all_data.append(dir_data)
    all_ids.extend(dir_ids)  # extend, not append

# Concatenate *all* raw data (across directories)
X_submission = np.concatenate(all_data, axis=0)
print("Shape of X_submission before scaling:", X_submission.shape)
X_submission = scaler.transform(X_submission)  # Scale using the training scaler
print("Shape of X_submission after scaling:", X_submission.shape)

# Reshape (AFTER all concatenation and scaling)
num_samples = len(X_submission) // 480
X_submission = X_submission.reshape(num_samples, 480, -1) # No truncation
X_submission = X_submission.astype('float32')

print("Final X_submission shape:", X_submission.shape)
print("Length of all_ids:", len(all_ids))  # all_ids, not id_submission

y_pred = model.predict(X_submission)
y_pred_classes = y_pred.argmax(axis=1)
y_pred_decoded = label_encoder.inverse_transform(y_pred_classes)

print("Length of y_pred_decoded:", len(y_pred_decoded))  # Should be num_samples
assert len(all_ids) == len(y_pred_decoded) == num_samples, "Final ID and prediction length mismatch!"


submission = pd.DataFrame({
    'id': all_ids,   # Use the correctly generated all_ids
    'labels': y_pred_decoded
})

submission.to_csv('submission.csv', index=False)
print("Submission file 'submission.csv' created successfully.")

X shape: (66473, 480, 8)
y_categorical shape: (66473, 3)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                      │ (None, 480, 32)             │             800 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization                  │ (None, 480, 32)             │             128 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling1d (MaxPooling1D)         │ (None, 240, 32)             │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv1d_1 (Conv1D)                    │ (None, 240, 64)             │           6,208 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_1                │ (None, 240, 64)             │             256 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling1d_1 (MaxPooling1D)       │ (None, 120, 64)             │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 120, 64)             │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ lstm (LSTM)                          │ (None, 120, 64)             │          33,024 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ lstm_1 (LSTM)                        │ (None, 64)                  │          33,024 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_1 (Dropout)                  │ (None, 64)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 128)                 │           8,320 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_2 (Dropout)                  │ (None, 128)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 3)                   │             387 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 82,147 (320.89 KB)

 Trainable params: 81,955 (320.14 KB)

 Non-trainable params: 192 (768.00 B)

Epoch 1/100
831/831 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.6279 - f1_score: 0.2146 - loss: 0.8925 - val_f1_score: 0.5451
831/831 ━━━━━━━━━━━━━━━━━━━━ 33s 30ms/step - accuracy: 0.6279 - f1_score: 0.2146 - loss: 0.8924 - val_accuracy: 0.6211 - val_f1_score: 0.5451 - val_loss: 0.8400
Epoch 2/100
830/831 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.6564 - f1_score: 0.1859 - loss: 0.8325 - val_f1_score: 0.5520
831/831 ━━━━━━━━━━━━━━━━━━━━ 21s 25ms/step - accuracy: 0.6564 - f1_score: 0.1860 - loss: 0.8325 - val_accuracy: 0.5625 - val_f1_score: 0.5520 - val_loss: 0.9069
Epoch 3/100
831/831 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 0.6724 - f1_score: 0.2999 - loss: 0.7937 - val_f1_score: 0.5379
831/831 ━━━━━━━━━━━━━━━━━━━━ 40s 25ms/step - accuracy: 0.6724 - f1_score: 0.2999 - loss: 0.7936 - val_accuracy: 0.5441 - val_f1_score: 0.5379 - val_loss: 0.9134
Epoch 4/100
831/831 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - accuracy: 0.6810 - f1_score: 0.3588 - loss: 0.7708 - val_f1_score: 0.51

In [3]:
set(submission['labels'])

{'N', 'W'}